In [ ]:
# =========================
# RQ7: Practical Usefulness
# Real-World Suitability Decision Matrix
# =========================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# -------------------------
# 1. Load Data
# -------------------------
df = pd.read_csv(
    "/kaggle/input/datasets/sharmajicoder/gaming-and-mental-health/gaming_mental_health_10M_40features.csv"
)

df = df.dropna()
df = df.sample(n=20000, random_state=42)

TARGET = df.columns[-1]

# -------------------------
# 2. Encode Target
# -------------------------
le = LabelEncoder()
y_raw = le.fit_transform(df[TARGET])

median_value = np.median(y_raw)
y = (y_raw > median_value).astype(int)

# -------------------------
# 3. Features
# -------------------------
X = pd.get_dummies(df.drop(TARGET, axis=1), drop_first=True).astype(float)

# -------------------------
# 4. Train/Test Split
# -------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# -------------------------
# 5. Define Models
# -------------------------
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, n_jobs=-1),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),
    "XGBoost": XGBClassifier(
        n_estimators=100,
        random_state=42,
        eval_metric="mlogloss",
        n_jobs=-1
    )
}

# -------------------------
# 6. Evaluate Models
# -------------------------
model_results = []

for name, model in models.items():
    print(f"Training {name}...")

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    model_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, average="weighted", zero_division=0),
        "Recall": recall_score(y_test, y_pred, average="weighted", zero_division=0),
        "F1": f1_score(y_test, y_pred, average="weighted", zero_division=0)
    })

performance_df = pd.DataFrame(model_results)
# performance_df.to_csv("RQ7_model_performance.csv", index=False)

# print("\n=== Model Performance ===")
# print(performance_df)

# -------------------------
# 7. Convert Metrics to 1–5 Scores
# -------------------------
def scale_to_1_5(series):
    if series.max() == series.min():
        return pd.Series([3] * len(series), index=series.index)
    return 1 + 4 * (series - series.min()) / (series.max() - series.min())

performance_df["Performance_Score"] = scale_to_1_5(performance_df["F1"]).round()

# -------------------------
# 8. Robustness Test
# -------------------------
robustness_results = []

for name, model in models.items():
    model.fit(X_train, y_train)

    baseline_pred = model.predict(X_test)
    baseline_f1 = f1_score(y_test, baseline_pred, average="weighted", zero_division=0)

    X_test_noise = X_test + np.random.normal(0, 0.1, X_test.shape)
    noise_pred = model.predict(X_test_noise)
    noise_f1 = f1_score(y_test, noise_pred, average="weighted", zero_division=0)

    robustness_ratio = noise_f1 / baseline_f1 if baseline_f1 != 0 else 0

    robustness_results.append({
        "Model": name,
        "Robustness_Ratio": robustness_ratio
    })

robustness_df = pd.DataFrame(robustness_results)
robustness_df["Robustness_Score"] = scale_to_1_5(
    robustness_df["Robustness_Ratio"]
).round()

# -------------------------
# 9. Practical Scores
# -------------------------
practical_scores = {
    "Logistic Regression": {
        "Interpretability": 5,
        "Cost Efficiency": 5,
        "Deployment Readiness": 5
    },
    "Random Forest": {
        "Interpretability": 3,
        "Cost Efficiency": 4,
        "Deployment Readiness": 4
    },
    "XGBoost": {
        "Interpretability": 2,
        "Cost Efficiency": 3,
        "Deployment Readiness": 3
    }
}

# -------------------------
# 10. Final Decision Matrix
# -------------------------
decision_rows = []

for model_name in models.keys():
    perf_score = performance_df.loc[
        performance_df["Model"] == model_name, "Performance_Score"
    ].values[0]

    robust_score = robustness_df.loc[
        robustness_df["Model"] == model_name, "Robustness_Score"
    ].values[0]

    decision_rows.append({
        "Model": model_name,
        "Performance": int(perf_score),
        "Interpretability": practical_scores[model_name]["Interpretability"],
        "Robustness": int(robust_score),
        "Cost Efficiency": practical_scores[model_name]["Cost Efficiency"],
        "Deployment Readiness": practical_scores[model_name]["Deployment Readiness"]
    })

decision_df = pd.DataFrame(decision_rows)
# decision_df.to_csv("RQ7_decision_matrix.csv", index=False)

# print("\n=== RQ7 Decision Matrix ===")
# print(decision_df)

# -------------------------
# 11. Weighted Score
# -------------------------
weights = {
    "Performance": 0.30,
    "Interpretability": 0.20,
    "Robustness": 0.20,
    "Cost Efficiency": 0.15,
    "Deployment Readiness": 0.15
}

decision_df["Weighted Score"] = (
    decision_df["Performance"] * weights["Performance"] +
    decision_df["Interpretability"] * weights["Interpretability"] +
    decision_df["Robustness"] * weights["Robustness"] +
    decision_df["Cost Efficiency"] * weights["Cost Efficiency"] +
    decision_df["Deployment Readiness"] * weights["Deployment Readiness"]
)

decision_df.to_csv("RQ7_final_table.csv", index=False)

print("\n=== Final RQ7 Table ===")
print(decision_df)

# -------------------------
# 12. Radar Chart
# -------------------------
criteria = [
    "Performance\n(Weight: 0.30)",
    "Interpretability\n(Weight: 0.20)",
    "Robustness\n(Weight: 0.20)",
    "Cost Efficiency\n(Weight: 0.15)",
    "Deployment Readiness\n(Weight: 0.15)"
]

criteria_clean = [
    "Performance",
    "Interpretability",
    "Robustness",
    "Cost Efficiency",
    "Deployment Readiness"
]

num_vars = len(criteria_clean)

angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
angles += angles[:1]

plt.figure(figsize=(9, 6))
ax = plt.subplot(111, polar=True)

for _, row in decision_df.iterrows():
    values = [row[c] for c in criteria_clean]
    values += values[:1]

    ax.plot(angles, values, marker="o", linewidth=2, label=row["Model"])
    ax.fill(angles, values, alpha=0.08)

ax.set_title(
    "RQ7: Practical Usefulness (Decision Matrix)",
    fontsize=14,
    fontweight="bold",
    pad=25
)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(criteria, fontsize=10)

ax.set_ylim(0, 5)
ax.set_yticks([1, 2, 3, 4, 5])
ax.set_yticklabels(["1", "2", "3", "4", "5"])

ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.10))

plt.tight_layout()
plt.savefig("RQ7_figure.pdf", bbox_inches="tight")
plt.show()